Pipeline falsa para probar el efecto de las variaciones de la imputacion en el desempeño de los modelos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
import os

from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

IMAGES_DIR = 'images'
os.makedirs(IMAGES_DIR, exist_ok=True)

def save_fig(name):
    path = os.path.join(IMAGES_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Figura guardada: {path}')

print('Setup OK  |  random_state =', RANDOM_STATE)

Setup OK  |  random_state = 42


In [ ]:
RAW_PATH = '../dataset.csv'
ENCODED_MISSING = {'No information','Does not apply','Does not apply ','does not apply','N/A',''}

def load_and_normalize(path):
    df = pd.read_csv(path)
    df = df[df['school'] != 'High school'].copy().reset_index(drop=True)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(
            lambda x: np.nan if (isinstance(x, str) and x.strip() in ENCODED_MISSING) else x)
    for col in ['admission.test', 'general.math.eval']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['regime'] = df['educational.model'].map({0: 'PreTec21', 1: 'Tec21'})
    return df

df = load_and_normalize(RAW_PATH)
print(f'Dataset: {df.shape}  |  PreTec21: {(df.regime=="PreTec21").sum():,}  |  Tec21: {(df.regime=="Tec21").sum():,}')
print(f'Tasa de deserción: {(df.retention==0).mean()*100:.1f}%')